# Creating Phylogenetic Tree

This notebook uses data from the GTDB to create to a phylogenetic tree:

## Key Features

- Analyzes top GDTB taxonomy classifications 
- Views which taxonomic classifications on our BacNasvs have more than 10 entries
- Creates a phylogenetic tree that includes families with BacNavs


In [2]:
!pip install seaborn


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# ============================================================
# GTDB TREE ANALYSIS
# FULL CSV + TOP100 PHYLA JSON + FUTURE QUERY SUPPORT
# ============================================================

from Bio import Phylo
from collections import Counter, defaultdict
import re
from tqdm import tqdm
import csv
import os
import json
import pandas as pd


# ============================================================
# LOAD TREE
# ============================================================

tree_file = "data_sources/bac120_r232.tree"
tree = Phylo.read(tree_file, "newick")

print("TREE LOADED")


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

out_csv = "results/phylotree_data/csv_data"
out_tree = "results/phylotree_data/tree_files"

os.makedirs(out_csv, exist_ok=True)
os.makedirs(out_tree, exist_ok=True)

csv_output = os.path.join(out_csv, "gtdb_full_taxonomy.csv")
json_output = os.path.join(out_tree, "top100_phyla_tree.json")


# ============================================================
# REGEX PATTERNS
# ============================================================

patterns = {
    "phylum": re.compile(r"p__([A-Za-z0-9_\-]+)"),
    "class":  re.compile(r"c__([A-Za-z0-9_\-]+)"),
    "order":  re.compile(r"o__([A-Za-z0-9_\-]+)"),
    "family": re.compile(r"f__([A-Za-z0-9_\-]+)"),
    "genus":  re.compile(r"g__([A-Za-z0-9_\-]+)")
}


# ============================================================
# CACHE NODE TAXONOMY
# ============================================================

node_tax = defaultdict(dict)

for clade in tree.find_clades():

    if not clade.name:
        continue

    for rank, regex in patterns.items():

        match = regex.search(clade.name)

        if match:
            node_tax[clade][rank] = match.group(1)

print("NODE CACHE READY")


# ============================================================
# STORAGE
# ============================================================

phylum = Counter()
cls = Counter()
order = Counter()
family = Counter()
genus = Counter()

rows = []


# ============================================================
# PROCESS TERMINALS
# ============================================================

terminals = tree.get_terminals()

print(f"Processing {len(terminals)} genomes...")


for tip in tqdm(terminals, desc="Scanning tree"):

    best = {
        "phylum": None,
        "class": None,
        "order": None,
        "family": None,
        "genus": None
    }

    # Walk root → tip path
    for node in tree.get_path(tip):

        if node in node_tax:

            for rank in best:

                if rank in node_tax[node]:
                    best[rank] = node_tax[node][rank]

    # ========================================================
    # STORE FULL GENOME LINEAGE
    # ========================================================

    row = {
        "genome": tip.name,
        "phylum": best["phylum"],
        "class": best["class"],
        "order": best["order"],
        "family": best["family"],
        "genus": best["genus"]
    }

    rows.append(row)

    # ========================================================
    # COUNTS
    # ========================================================

    if best["phylum"]:
        phylum[best["phylum"]] += 1

    if best["class"]:
        cls[best["class"]] += 1

    if best["order"]:
        order[best["order"]] += 1

    if best["family"]:
        family[best["family"]] += 1

    if best["genus"]:
        genus[best["genus"]] += 1


# ============================================================
# CREATE DATAFRAME (IMPORTANT FOR FUTURE ANALYSIS)
# ============================================================

taxonomy_df = pd.DataFrame(rows)

print("\nDATAFRAME CREATED")


# ============================================================
# SAVE FULL CSV
# ============================================================

taxonomy_df.to_csv(csv_output, index=False)

print(f"CSV SAVED → {csv_output}")


# ============================================================
# TOP 100 PHYLA
# ============================================================

top_phyla = set([p for p, _ in phylum.most_common(100)])


# ============================================================
# BUILD HIERARCHICAL TAXONOMY TREE
# ============================================================

tree_dict = {}

for row in rows:

    p = row["phylum"]
    c = row["class"]
    o = row["order"]
    f = row["family"]
    g = row["genus"]

    # Skip non-top100 phyla
    if not p or p not in top_phyla:
        continue

    # ========================================================
    # PHYLUM
    # ========================================================

    tree_dict.setdefault(p, {
        "count": 0,
        "classes": {}
    })

    tree_dict[p]["count"] += 1

    # ========================================================
    # CLASS
    # ========================================================

    if c:

        tree_dict[p]["classes"].setdefault(c, {
            "count": 0,
            "orders": {}
        })

        tree_dict[p]["classes"][c]["count"] += 1

        # ====================================================
        # ORDER
        # ====================================================

        if o:

            tree_dict[p]["classes"][c]["orders"].setdefault(o, {
                "count": 0,
                "families": {}
            })

            tree_dict[p]["classes"][c]["orders"][o]["count"] += 1

            # ================================================
            # FAMILY
            # ================================================

            if f:

                tree_dict[p]["classes"][c]["orders"][o]["families"].setdefault(f, {
                    "count": 0,
                    "genera": {}
                })

                tree_dict[p]["classes"][c]["orders"][o]["families"][f]["count"] += 1

                # ============================================
                # GENUS
                # ============================================

                if g:

                    genera = tree_dict[p]["classes"][c]["orders"][o]["families"][f]["genera"]

                    genera[g] = genera.get(g, 0) + 1


# ============================================================
# SAVE JSON TREE
# ============================================================

with open(json_output, "w") as f:
    json.dump(tree_dict, f, indent=2)

print(f"JSON TREE SAVED → {json_output}")


# ============================================================
# PRINT TOP 10 COUNTS
# ============================================================

def show(title, counter):

    print("\n" + title)
    print("=" * 50)

    for k, v in counter.most_common(10):
        print(f"{k}: {v}")


show("TOP PHYLA", phylum)
show("TOP CLASSES", cls)
show("TOP ORDERS", order)
show("TOP FAMILIES", family)
show("TOP GENERA", genus)


# ============================================================
# FUTURE ANALYSIS EXAMPLES
# ============================================================

print("\nEXAMPLE FUTURE ANALYSIS COMMANDS:")
print("=" * 50)

print("""
# Top 20 classes
taxonomy_df["class"].value_counts().head(20)

# Top 30 genera
taxonomy_df["genus"].value_counts().head(30)

# Count genera per phylum
taxonomy_df.groupby("phylum")["genus"].nunique()

# Count families per order
taxonomy_df.groupby("order")["family"].nunique()

# Filter one phylum
taxonomy_df[taxonomy_df["phylum"] == "Proteobacteria"]
""")


print("\nDONE — FULL TAXONOMY PIPELINE COMPLETE")

TREE LOADED
NODE CACHE READY
Processing 189801 genomes...


Scanning tree: 100%|████████████████████████████████████████████████████████| 189801/189801 [24:26:45<00:00,  2.16it/s]



DATAFRAME CREATED
CSV SAVED → results/phylotree_data/csv_data\gtdb_full_taxonomy.csv
JSON TREE SAVED → results/phylotree_data/tree_files\top100_phyla_tree.json

TOP PHYLA
Pseudomonadota: 46828
Bacillota: 27167
Actinomycetota: 24968
Bacteroidota: 23901
Patescibacteriota: 10945
Acidobacteriota: 7453
Chloroflexota: 6831
Planctomycetota: 4995
Verrucomicrobiota: 4825
Desulfobacterota: 4552

TOP CLASSES
Gammaproteobacteria: 25165
Bacteroidia: 21942
Alphaproteobacteria: 21473
Clostridia: 20027
Actinomycetes: 14793
Bacilli: 4864
Minisyncoccia: 4322
Terriglobia: 4081
Bacilli_A: 3799
Verrucomicrobiia: 3767

TOP ORDERS
Burkholderiales: 9155
Bacteroidales: 8804
Oscillospirales: 7815
Rhizobiales: 5752
Lachnospirales: 4973
Pseudomonadales: 4700
Flavobacteriales: 4674
Actinomycetales: 4579
Mycobacteriales: 3781
Chitinophagales: 3446

TOP FAMILIES
Lachnospiraceae: 4549
Flavobacteriaceae: 3764
Rhodobacteraceae: 3226
Burkholderiaceae_C: 2966
Sphingomonadaceae: 2805
Streptomycetaceae: 2611
Acutalibacter

In [17]:
import pandas as pd

csv_path = "data_sources/vgsc-id90_min100_max500_merged_sequence_taxonomy.csv"

# Read only the header row
df = pd.read_csv(csv_path, nrows=0)

# Print column names
print(df.columns.tolist())

['id', 'taxonomy_id', 'organism_from_ncb', 'subspecies', 'species', 'genus', 'family', 'order', 'class', 'phylum', 'kingdom', 'domain', 'cellular_root', 'sequence']


In [18]:
# ============================================================
# COMPARE TAXONOMY BETWEEN:
# 1. GTDB TREE TAXONOMY CSV
# 2. SEQUENCE TAXONOMY CSV
#
# GOAL:
# - Find taxonomy classifications appearing >=10 times
# - Compare overlap/shared taxonomy
# - Print shared classifications
# ============================================================

import pandas as pd
from collections import Counter
import os


# ============================================================
# INPUT FILES
# ============================================================

gtdb_csv = "results/phylotree_data/csv_data/gtdb_full_taxonomy.csv"

sequence_csv = (
    "data_sources/"
    "vgsc-id90_min100_max500_merged_sequence_taxonomy.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

print("LOADING CSV FILES...")

gtdb_df = pd.read_csv(gtdb_csv)
seq_df = pd.read_csv(sequence_csv)

print("CSV FILES LOADED")


# ============================================================
# TAXONOMY RANKS TO ANALYZE
# ============================================================

taxonomy_ranks = [
    "phylum",
    "class",
    "order",
    "family",
    "genus"
]


# ============================================================
# MINIMUM OCCURRENCE THRESHOLD
# ============================================================

MIN_COUNT = 10


# ============================================================
# STORE RESULTS
# ============================================================

shared_results = {}


# ============================================================
# ANALYZE EACH TAXONOMIC RANK
# ============================================================

for rank in taxonomy_ranks:

    print("\n" + "=" * 70)
    print(f"ANALYZING: {rank.upper()}")
    print("=" * 70)

    # ========================================================
    # COUNTS IN SEQUENCE CSV
    # ========================================================

    seq_counts = (
        seq_df[rank]
        .dropna()
        .value_counts()
    )

    # Keep only taxa appearing >= MIN_COUNT
    seq_valid = seq_counts[seq_counts >= MIN_COUNT]

    print(f"\n{rank.upper()} entries appearing >= {MIN_COUNT} times:")
    print("-" * 50)

    for taxon, count in seq_valid.items():
        print(f"{taxon}: {count}")

    # ========================================================
    # COUNTS IN GTDB CSV
    # ========================================================

    gtdb_counts = (
        gtdb_df[rank]
        .dropna()
        .value_counts()
    )

    gtdb_taxa = set(gtdb_counts.index)
    seq_taxa = set(seq_valid.index)

    # ========================================================
    # FIND SHARED TAXA
    # ========================================================

    shared = seq_taxa.intersection(gtdb_taxa)

    shared_results[rank] = shared

    print(f"\nSHARED {rank.upper()} BETWEEN DATASETS:")
    print("-" * 50)

    if len(shared) == 0:
        print("NONE FOUND")

    else:

        for taxon in sorted(shared):

            seq_count = seq_counts[taxon]
            gtdb_count = gtdb_counts.get(taxon, 0)

            print(
                f"{taxon} | "
                f"Sequence CSV: {seq_count} | "
                f"GTDB Tree: {gtdb_count}"
            )


# ============================================================
# OPTIONAL:
# SAVE SHARED RESULTS
# ============================================================

out_dir = "results/phylotree_data/csv_data"
os.makedirs(out_dir, exist_ok=True)

shared_output = os.path.join(
    out_dir,
    "shared_taxonomy_comparison.csv"
)

rows = []

for rank, taxa in shared_results.items():

    for taxon in taxa:

        rows.append({
            "rank": rank,
            "taxon": taxon
        })

shared_df = pd.DataFrame(rows)

shared_df.to_csv(shared_output, index=False)

print("\n" + "=" * 70)
print("SHARED TAXONOMY CSV SAVED")
print(shared_output)
print("=" * 70)


# ============================================================
# EXAMPLE FUTURE QUERIES
# ============================================================

print("""
FUTURE ANALYSIS EXAMPLES
============================================================

# Top 20 shared genera
shared_df[shared_df["rank"] == "genus"]

# Count overlap by rank
shared_df["rank"].value_counts()

# Filter one phylum in sequence data
seq_df[seq_df["phylum"] == "Proteobacteria"]

# Compare genus abundance
seq_df["genus"].value_counts().head(20)

# Compare GTDB abundance
gtdb_df["genus"].value_counts().head(20)
""")

LOADING CSV FILES...
CSV FILES LOADED

ANALYZING: PHYLUM

PHYLUM entries appearing >= 10 times:
--------------------------------------------------
Pseudomonadota: 501
Actinomycetota: 215
Bacillota: 79
Bacteroidota: 15

SHARED PHYLUM BETWEEN DATASETS:
--------------------------------------------------
Actinomycetota | Sequence CSV: 215 | GTDB Tree: 24968
Bacillota | Sequence CSV: 79 | GTDB Tree: 27167
Bacteroidota | Sequence CSV: 15 | GTDB Tree: 23901
Pseudomonadota | Sequence CSV: 501 | GTDB Tree: 46828

ANALYZING: CLASS

CLASS entries appearing >= 10 times:
--------------------------------------------------
Alphaproteobacteria: 338
Actinomycetes: 215
Gammaproteobacteria: 160
Bacilli: 74

SHARED CLASS BETWEEN DATASETS:
--------------------------------------------------
Actinomycetes | Sequence CSV: 215 | GTDB Tree: 14793
Alphaproteobacteria | Sequence CSV: 338 | GTDB Tree: 21473
Bacilli | Sequence CSV: 74 | GTDB Tree: 4864
Gammaproteobacteria | Sequence CSV: 160 | GTDB Tree: 25165

ANA

In [23]:
# ============================================================
# SHARED TAXONOMY TREE BUILDER
# WITH TAXONOMY NAME NORMALIZATION / MERGING
#
# GOAL
# ----
# Merge near-duplicate taxonomy names like:
#
# Bacilli
# Bacilli_A
#
# into:
#
# Bacilli
#
# BEFORE:
# - shared comparison
# - abundance filtering
# - tree building
#
# ============================================================

import pandas as pd
import json
import os
import re


# ============================================================
# PARAMETERS
# ============================================================

MIN_SHARED_COUNT = 10
MIN_BRANCHES = 15

TARGET_RANK = "class"

# If True:
# Bacilli_A -> Bacilli
# Bacilli_B -> Bacilli
# etc.
NORMALIZE_SUFFIXES = True


# ============================================================
# INPUT FILES
# ============================================================

gtdb_csv = (
    "results/phylotree_data/csv_data/"
    "gtdb_full_taxonomy.csv"
)

sequence_csv = (
    "data_sources/"
    "vgsc-id90_min100_max500_merged_sequence_taxonomy.csv"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

out_dir = "results/phylotree_data/shared_taxonomy_trees"
os.makedirs(out_dir, exist_ok=True)

json_output = os.path.join(
    out_dir,
    f"shared_{TARGET_RANK}_tree.json"
)

newick_output = os.path.join(
    out_dir,
    f"shared_{TARGET_RANK}_tree.nwk"
)


# ============================================================
# TAXONOMY ORDER
# ============================================================

taxonomy_order = [
    "phylum",
    "class",
    "order",
    "family",
    "genus"
]

target_idx = taxonomy_order.index(TARGET_RANK)

used_ranks = taxonomy_order[:target_idx + 1]


# ============================================================
# NORMALIZATION FUNCTION
# ============================================================

def normalize_taxonomy_name(name):

    """
    Convert:
        Bacilli_A -> Bacilli
        Bacilli_B -> Bacilli

    Keeps:
        Alphaproteobacteria unchanged
    """

    if pd.isna(name):
        return name

    name = str(name)

    if NORMALIZE_SUFFIXES:

        # Remove trailing _A, _B, _C etc.
        name = re.sub(r"_[A-Z]+$", "", name)

    return name


# ============================================================
# LOAD CSV FILES
# ============================================================

print("\nLOADING CSV FILES...")

gtdb_df = pd.read_csv(gtdb_csv)
seq_df = pd.read_csv(sequence_csv)

print("FILES LOADED")


# ============================================================
# NORMALIZE TAXONOMY NAMES
# ============================================================

print("\nNORMALIZING TAXONOMY NAMES...")

for rank in taxonomy_order:

    if rank in gtdb_df.columns:
        gtdb_df[rank] = gtdb_df[rank].apply(
            normalize_taxonomy_name
        )

    if rank in seq_df.columns:
        seq_df[rank] = seq_df[rank].apply(
            normalize_taxonomy_name
        )

print("NORMALIZATION COMPLETE")


# ============================================================
# FIND TAXA >= MIN_SHARED_COUNT
# ============================================================

seq_counts = (
    seq_df[TARGET_RANK]
    .dropna()
    .value_counts()
)

valid_seq_taxa = set(
    seq_counts[seq_counts >= MIN_SHARED_COUNT].index
)

print("\nTAXA PASSING THRESHOLD")
print("=" * 60)

if len(valid_seq_taxa) == 0:

    print(
        f"\nNO {TARGET_RANK} entries appear "
        f"{MIN_SHARED_COUNT} or more times."
    )

    print("\nNO FILES CREATED")

    raise SystemExit

for taxon in sorted(valid_seq_taxa):

    print(f"{taxon}: {seq_counts[taxon]}")


# ============================================================
# FIND SHARED TAXA
# ============================================================

gtdb_counts = (
    gtdb_df[TARGET_RANK]
    .dropna()
    .value_counts()
)

gtdb_taxa = set(gtdb_counts.index)

shared_taxa = valid_seq_taxa.intersection(gtdb_taxa)

print("\nSHARED TAXA")
print("=" * 60)

if len(shared_taxa) == 0:

    print("\nNO SHARED TAXA FOUND")
    print("NO FILES CREATED")

    raise SystemExit

for taxon in sorted(shared_taxa):

    print(
        f"{taxon} | "
        f"SEQ={seq_counts[taxon]} | "
        f"GTDB={gtdb_counts[taxon]}"
    )


# ============================================================
# FILL REMAINING BRANCHES
# ============================================================

final_taxa = set(shared_taxa)

if len(final_taxa) < MIN_BRANCHES:

    needed = MIN_BRANCHES - len(final_taxa)

    print("\nADDING MOST ABUNDANT GTDB TAXA")
    print("=" * 60)

    for taxon, count in gtdb_counts.items():

        if taxon not in final_taxa:

            final_taxa.add(taxon)

            print(f"ADDED: {taxon} ({count})")

            needed -= 1

            if needed == 0:
                break


# ============================================================
# FINAL TAXA
# ============================================================

print("\nFINAL TAXA USED")
print("=" * 60)

for taxon in sorted(final_taxa):
    print(taxon)


# ============================================================
# FILTER GTDB DATAFRAME
# ============================================================

filtered_df = gtdb_df[
    gtdb_df[TARGET_RANK].isin(final_taxa)
].copy()


# ============================================================
# BUILD HIERARCHICAL TREE
# ============================================================

tree_dict = {}

for _, row in filtered_df.iterrows():

    current = tree_dict

    for rank in used_ranks:

        value = row.get(rank)

        if pd.isna(value):
            continue

        if value not in current:
            current[value] = {}

        current = current[value]


# ============================================================
# SAVE JSON TREE
# ============================================================

with open(json_output, "w") as f:
    json.dump(tree_dict, f, indent=2)

print("\nJSON TREE SAVED")
print(json_output)


# ============================================================
# CONVERT TO NEWICK
# ============================================================

def dict_to_newick(d):

    parts = []

    for key, value in d.items():

        if isinstance(value, dict) and len(value) > 0:

            subtree = dict_to_newick(value)

            parts.append(f"({subtree}){key}")

        else:
            parts.append(str(key))

    return ",".join(parts)


newick_str = f"({dict_to_newick(tree_dict)});"


# ============================================================
# SAVE NEWICK TREE
# ============================================================

with open(newick_output, "w") as f:
    f.write(newick_str)

print("\nNEWICK TREE SAVED")
print(newick_output)


# ============================================================
# SUMMARY
# ============================================================

print("\nSUMMARY")
print("=" * 60)

print(f"TARGET RANK: {TARGET_RANK}")
print(f"MIN_SHARED_COUNT: {MIN_SHARED_COUNT}")
print(f"MIN_BRANCHES: {MIN_BRANCHES}")

print(f"\nTOTAL SHARED TAXA: {len(shared_taxa)}")
print(f"TOTAL FINAL TAXA: {len(final_taxa)}")

print("\nDONE")


LOADING CSV FILES...
FILES LOADED

NORMALIZING TAXONOMY NAMES...
NORMALIZATION COMPLETE

TAXA PASSING THRESHOLD
Actinomycetes: 215
Alphaproteobacteria: 338
Bacilli: 74
Gammaproteobacteria: 160

SHARED TAXA
Actinomycetes | SEQ=215 | GTDB=14793
Alphaproteobacteria | SEQ=338 | GTDB=21473
Bacilli | SEQ=74 | GTDB=8663
Gammaproteobacteria | SEQ=160 | GTDB=25165

ADDING MOST ABUNDANT GTDB TAXA
ADDED: Bacteroidia (21942)
ADDED: Clostridia (20027)
ADDED: Minisyncoccia (4322)
ADDED: Terriglobia (4081)
ADDED: Verrucomicrobiia (3767)
ADDED: Thermoleophilia (3637)
ADDED: Acidimicrobiia (3543)
ADDED: Anaerolineae (2674)
ADDED: Cyanobacteriia (2402)
ADDED: Planctomycetia (2132)
ADDED: Coriobacteriia (2095)

FINAL TAXA USED
Acidimicrobiia
Actinomycetes
Alphaproteobacteria
Anaerolineae
Bacilli
Bacteroidia
Clostridia
Coriobacteriia
Cyanobacteriia
Gammaproteobacteria
Minisyncoccia
Planctomycetia
Terriglobia
Thermoleophilia
Verrucomicrobiia

JSON TREE SAVED
results/phylotree_data/shared_taxonomy_trees\sh